# Client Testing Notebook
Test each external client independently: Census Geocoder, US Congress API, Cicero, Google Civic, Tavily, and Anthropic.

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv

load_dotenv("../.env")

print("GOOGLE_CIVIC_API_KEY:", "set" if os.environ.get("GOOGLE_CIVIC_API_KEY") else "MISSING")
print("TAVILY_API_KEY:", "set" if os.environ.get("TAVILY_API_KEY") else "MISSING")
print("ANTHROPIC_API_KEY:", "set" if os.environ.get("ANTHROPIC_API_KEY") else "MISSING")
print("CICERO_API_KEY:", "set" if os.environ.get("CICERO_API_KEY") else "MISSING")
print("US_CONGRESS_API_KEY:", "set" if os.environ.get("US_CONGRESS_API_KEY") else "MISSING")

## 1. Census Geocoder (address → state + congressional district)
Free, no API key required. Used to resolve an address into state abbreviation and congressional district number.

In [ ]:
TEST_ADDRESS = "350 Fifth Avenue, New York, NY 10118"
CURRENT_CONGRESS = 119

resp_census = httpx.get(
    "https://geocoding.geo.census.gov/geocoder/geographies/onelineaddress",
    params={
        "address": TEST_ADDRESS,
        "benchmark": "Public_AR_Current",
        "vintage": "Current_Current",
        "format": "json",
    },
    timeout=15,
)
print(f"Status: {resp_census.status_code}")

matches = resp_census.json().get("result", {}).get("addressMatches", [])
if not matches:
    print("ERROR: No address match found")
else:
    geos = matches[0]["geographies"]
    state_abbr = geos["States"][0]["STUSAB"]
    cd_key = f"{CURRENT_CONGRESS}th Congressional Districts"
    district_num = geos[cd_key][0][f"CD{CURRENT_CONGRESS}"]
    print(f"State: {state_abbr}")
    print(f"Congressional District: {district_num}")
    print(f"\nWill query Congress API for: /member/congress/{CURRENT_CONGRESS}/{state_abbr}/{district_num}")

## 2. US Congress API (federal representatives)
Uses state + district from Census geocoder to fetch senators and House rep. Rate limit: 20,000/day.

In [ ]:
CONGRESS_API_URL = "https://api.congress.gov/v3"
api_key = os.environ["US_CONGRESS_API_KEY"]

# Fetch all members for this state in the current congress
resp_members = httpx.get(
    f"{CONGRESS_API_URL}/member/congress/{CURRENT_CONGRESS}/{state_abbr}",
    params={"api_key": api_key, "format": "json", "limit": 50},
    timeout=15,
)
print(f"Status: {resp_members.status_code}")

all_members = resp_members.json().get("members", [])
print(f"Total members for {state_abbr} in {CURRENT_CONGRESS}th Congress: {len(all_members)}\n")

# Filter to senators + matching district
for m in all_members:
    m_dist = m.get("district")
    if m_dist is None or str(m_dist) == str(district_num):
        label = "Senator" if m_dist is None else f"House District {m_dist}"
        print(f"  {m['name']} — {label} — {m['partyName']}")

In [ ]:
# Fetch full detail for one member
sample_member = [m for m in all_members if m.get("district") is None][0]  # first senator
resp_detail = httpx.get(
    sample_member["url"],
    params={"api_key": api_key, "format": "json"},
    timeout=15,
)
detail = resp_detail.json().get("member", {})

print(f"Name: {detail.get('directOrderName')}")
print(f"Party: {detail.get('partyHistory', [{}])[-1].get('partyName')}")
print(f"Photo: {detail.get('depiction', {}).get('imageUrl')}")
print(f"Phone: {detail.get('addressInformation', {}).get('phoneNumber')}")
print(f"Website: {detail.get('officialWebsiteUrl')}")
print(f"Leadership: {detail.get('leadership', [])}")
print(f"Sponsored bills: {detail.get('sponsoredLegislation', {}).get('count')}")
print(f"Current member: {detail.get('currentMember')}")

## 3. Google Civic Information API

In [ ]:
ELECTIONS_URL = "https://www.googleapis.com/civicinfo/v2/elections"

resp_elections = httpx.get(
    ELECTIONS_URL,
    params={"key": os.environ["GOOGLE_CIVIC_API_KEY"]},
    timeout=15,
)

print(f"Status: {resp_elections.status_code}")
if resp_elections.status_code != 200:
    print(f"Error: {resp_elections.text}")
    print()
    print("If this also returns 404 'Method not found', the Google Civic Information API")
    print("is NOT enabled in your GCP project. Go to:")
    print("  https://console.cloud.google.com/apis/library/civicinfo.googleapis.com")
    print("and click 'Enable'.")
else:
    data = resp_elections.json()
    print(f"Elections found: {len(data.get('elections', []))}")
    print(json.dumps(data, indent=2))

## 4. Tavily Search API

In [ ]:
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

results = tavily.search(query="US Senator from California", max_results=3)

print(f"Results: {len(results.get('results', []))}")
for r in results.get("results", []):
    print(f"  - {r['title']}")
    print(f"    {r['url']}")
    print(f"    {r['content'][:150]}...")
    print()

## 5. Anthropic (Claude) API

In [ ]:
import anthropic

client = anthropic.Anthropic()

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=256,
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
)

print(f"Stop reason: {message.stop_reason}")
print(f"Response: {message.content[0].text}")